

Step	学習内容	理解すること
1	Motion Subspace	関節運動を6次元Spatial Vectorで表す
2	Spatial Velocityの再帰計算	親リンクから子リンクへ速度を伝播
3	Spatial Accelerationの再帰計算	関節加速度とコリオリ項を考慮
4	Spatial Forceの計算	Spatial Newton-Euler式を利用
5	RNEAのBackward Recursion	子リンクから親リンクへ力を伝播し、関節トルクを計算
6	2リンクアームで検証	以前Kane法で導出した運動方程式と比較

## 1. Motion Subspace

Spatial Velocityは角速度と並進速度を用いて以下である。

$$
\boldsymbol{V} = \begin{bmatrix} \boldsymbol{\omega} \\ \boldsymbol{v} \end{bmatrix} \in \mathbf{R}^{6 \times 1}
$$

以下の親リンクにたいする関節の相対運動として、$z_i$軸に回転自由度がある剛体の Spatial Velocity を考える。

<p align="center">
<img src="./images/1DoF_revolute_joint.png" style="width:34%;" />
</p>

この場合、基準点を関節部の$J$とすれば、角速度 $\boldsymbol{\omega}$ と並進速度 $\boldsymbol{v}_J$ は以下となる。

$$
\boldsymbol{\omega} = \begin{bmatrix} 0 \\ 0 \\ \dot{q} \end{bmatrix}, \quad \boldsymbol{v}_J = \begin{bmatrix} 0 \\ 0 \\ 0 \end{bmatrix}
$$

よって、Spatial Velocityは以下となる。

$$
\boldsymbol{V}_J = \begin{bmatrix} \boldsymbol{\omega} \\ \boldsymbol{v}_J \end{bmatrix} =
\begin{bmatrix} 0 \\ 0 \\ \dot{q} \\ 0 \\ 0 \\ 0 \end{bmatrix} =
\underbrace{\begin{bmatrix} 0 \\ 0 \\ 1 \\ 0 \\ 0 \\ 0 \end{bmatrix}}_{\boldsymbol{S}} \dot{q}
$$

上式の$\boldsymbol{S}$を Motion Subspace と呼び、関節の一般化速度をSpatial Velicityに変換する写像と理解できる。

$$
\boldsymbol{V}_J = \boldsymbol{S} \dot{q}
$$


### RNEA とSubspaceとの関連

RNEAでは各リンクのSpatial Velocityを次のように計算する。

$$
\boldsymbol{V}_i = {^i}X_{\lambda(i)} \ \boldsymbol{V}_{\lambda(i)} + \boldsymbol{S}_i \dot{q}_i
$$

ここで、

- $\lambda(i)$ : リンク$i$の親リンク
- ${^i}X_{\lambda(i)}$ : 親リンク$\lambda(i)$のSpatial Velocityを子リンク$i$座標系へ変換する行列
- $\boldsymbol{S}_i \dot{q}_i$ : 親リンクに対する相対運動

さらにSpatial Force Vector $\boldsymbol{f}_i$ から以下のように関節トルクをSubspace で取り出すことが出来る。

$$
\tau_i = \boldsymbol{S}_i^T \boldsymbol{f}_i
$$

この場合、Motion Subspaceは関節の一般化速度からSpatial Velocityを作るだけでなく、Spatial Force Vector から関節の一般化力を取り出す役割も持つ。

これは以下のように導くことが出来る。

Spatial Power(仕事率)は以下で計算できる。

$$
P = \boldsymbol{f}^T \boldsymbol{V}
$$

関節の相対速度を$\boldsymbol{V} = S \dot{q}$ とすると、以下となる。

$$
P = \boldsymbol{f}^T S \dot{q} = (S^T \boldsymbol{f}) \dot{q}
$$

一方、一般化力$\tau$と一般化速度$\dot{q}$により仕事率は以下である。

$$
P = \tau^T \dot{q}
$$

したがって、任意の$\dot{q}$に対して仕事率が一致するためには、以下の関係が必要である。

$$
\tau = S^T \boldsymbol{f}
$$


## 2. Spatial Velocityの再帰計算(Forward Recursion)

### 2.1 親リンクと子リンクの速度の関係

親リンクを$i-1$、子リンクを$i$とする。

子リンクの運動は次の２つから構成される。

1. 親リンクの運動によって生じる速度
2. 親リンクに対する子リンクの相対運動によって生じる速度

よって子リンクのSpatial Velocityは次のように計算できる。

$$
\boxed{
{^i}V_{i,O_i} = {^i}X_{i-1, O_{i-1} \rightarrow O_i} \ {}^{i-1}\boldsymbol{V}_{i-1, O_{i-1}} + {^i}\boldsymbol{S}_{i, O_i} \dot{q}_i
}
$$

各項の意味は以下となる。

- ${^i}V_{i,O_i}$ : 子リンク$i$の$i$座標系の基準点$O_i$におけるSpatial Velocity
- ${^i}X_{i-1, O_{i-1} \rightarrow O_i}$ : 親リンクのSpatial Velocityを子リンクの座標系へ変換し、基準点を$O_{i-1}$から$O_i$へ変更するSpatial Transform
- ${i-1}\boldsymbol{V}_{i-1, O_{i-1}}$ : 親リンク$i-1$の$i-1$座標系の基準点$O_{i-1}$におけるSpatial Velocity
- ${^i}\boldsymbol{S}_{i, O_i} \dot{q}_i$ : 親リンク$i-1$に対する子リンク$i$の基準点$O_i$における相対Spatial Velocity

RNEAでは各リンクに剛体座標系を設定し、Spatial Velocityをその座標系の原点で表す約束がある。<br>
また、Motion Subspaceにも基準点が必要であり、例えばリンク$i$の関節がz軸で回転し、その軸が$O_i$を通るなら、Motion Subspaceは以下となる。

$$
{^i}\boldsymbol{S}_{i, O_i} = \begin{bmatrix} 0 & 0 & 1 & 0 & 0 & 0 \end{bmatrix}^T
$$

しかし、基準点$P$が関節軸$O_i$から離れている場合、以下のようになる。

$$
{^i}\boldsymbol{S}_{i, P} = \begin{bmatrix} e_z \\ e_z \times {^i}\boldsymbol{r}_{O_i P} \end{bmatrix}
$$

よって、座標系とその原点は次のように設定を行う。

- リンクが回転する場合、回転する軸に座標系のある軸を合わせるように設定する。原点は合わせた軸上となるが、その位置は親リンクの原点から適切な位置とする。
- リンクが直動する場合、並進する方向に座標系のある軸を合わせるように設定する。原点の位置は適切な位置とする。

例えば、親リンク$i-1$と子リンク$i$が下の図のように構成されており、親リンクは$v_{i-1}$方向に並進し、子リンクは$O_i$まわりに回転しているとする。親座標系は$v_{i-1}$と$x_{i-1}$が一致するように設定し、その原点は剛体中心としている。子座標系は回転軸と$z_i$方向を一致するように設定し、原点を回転軸上としている。

<p align="center">
<img src="./images/rnea-1.png" style="width:40%;" />
</p>

RNEAでは、それぞれのリンクの運動の基準点をそのリンクの座標原点で示す約束となるため、今後は上式から基準点の情報を省略した以下の記述を行う。

改めて、子リンクのSpatial Velocityの計算式は以下と再帰式となる。

$$
\boxed{
{^i}\boldsymbol{V}_{i} = {^i}X_{i-1} \ {^{i-1}}\boldsymbol{V}_{i-1} + {^i}\boldsymbol{S}_{i} \dot{q}_i
}
$$

- ${^i}\boldsymbol{V}_{i}$ : リンク$i$座標で示したリンク$i$のSpatial Velocity
- ${^i}X_{i-1}$ : 親リンクのSpatial Velocityを子リンクの座標系・基準点へ変換するSpatial Transform
- ${^i}\boldsymbol{S}_{i} \dot{q}_i$ : 親リンクに対する子リンクの相対 Spatial Velocity

### 2.2 Spatial Transform の展開

[このNote](./spatial_vector.ipynb)の2節でSpatial VelocityのSpatial Transformを導いた。RENAでは親リンクの原点$O_{i-1}$から子リンクの原点$O_i$への基準点変換を行う。$O_{i-1}$から$O_i$までの位置ベクトルを子座標で表したものを以下とする。

$$
{}^i \boldsymbol{r}_{i-1, i}
$$

Spatial Transformは以下となる。

$$
{^i}X_{i-1} = \begin{bmatrix}
{}^i R _{i-1} & 0 \\
-[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1} & {}^i R _{i-1}
\end{bmatrix}
$$

再帰式に代入すると以下となる。

$$
\begin{bmatrix} {}^i \boldsymbol{\omega}_i \\ {}^i \boldsymbol{v}_i \end{bmatrix} =
\begin{bmatrix}
{}^i R _{i-1} & 0 \\
-[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1} & {}^i R _{i-1}
\end{bmatrix}
\begin{bmatrix} {}^{i-1} \boldsymbol{\omega}_{i-1} \\ {}^{i-1} \boldsymbol{v}_{i-1} \end{bmatrix} +
{}^i \boldsymbol{S}_i \dot{q}_i
$$

### 2.3 具体例

上で示した図の子リンクのSpatial Velocityを求めてみる。

子リンクは子リンク座標系の$z_i$軸まわりに$\dot{q}_i$で回転するとする。そのため、Motion Subspaceは以下となる。

$$
{}^i \boldsymbol{S}_i = \begin{bmatrix} 0 & 0 & 1 & 0 & 0 & 0 \end{bmatrix}^T
$$

これより、子リンクのSpatial Velcotyは以下となる。

$$
\begin{bmatrix} {}^i \boldsymbol{\omega}_i \\ {}^i \boldsymbol{v}_i \end{bmatrix} =
\begin{bmatrix}
{}^i R _{i-1} & 0 \\
-[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1} & {}^i R _{i-1}
\end{bmatrix}
\begin{bmatrix} {}^{i-1} \boldsymbol{\omega}_{i-1} \\ {}^{i-1} \boldsymbol{v}_{i-1} \end{bmatrix} +
\begin{bmatrix} 0 \\ 0 \\ \dot{q}_i \\ 0 \\ 0 \\ 0 \end{bmatrix} 
$$

角速度と並進速度に分解する。

- 角速度

    $$
    {}^i \boldsymbol{\omega}_i = {}^i R _{i-1} {}^{i-1} \boldsymbol{\omega}_{i-1} + \begin{bmatrix} 0 \\ 0 \\ \dot{q}_i \end{bmatrix} 
    $$

    この式は親リンクの角速度を子リンク座標へ変換したものと、相対角速度の合成であることが分かる。

- 並進速度

    $$
    \begin{aligned}
    {}^i \boldsymbol{v}_i &= -[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1}{}^{i-1} \boldsymbol{\omega}_{i-1} + {}^i R _{i-1}{}^{i-1} \boldsymbol{v}_{i-1} \\
    &= {}^i R _{i-1}{}^{i-1} \boldsymbol{v}_{i-1} + ({}^i R_{i-1}{}^{i-1} \boldsymbol{\omega}_{i-1}) \times {}^i \boldsymbol{r}_{i-1, i}
    \end{aligned}
    $$

    ${}^i \boldsymbol{v}_i$は子リンク原点のSpatial Velocityである。よって、親リンクの並進速度と、親リンク原点と子リンク原点までの位置ベクトルを親リンクの角速度で回転させた接線速度との合成であることが分かる。
    
### 2.4 再帰計算について

例えば、3リンクアームの場合、固定ベースのSpatial Velocityを以下とする。

$$
{}^0 \boldsymbol{V}_0 = 0
$$

すると、2.1節で示したSpatial Velocityの再帰式より、リンク1からリンク2, リンク3へと順番に計算ができる。

$$
\begin{aligned}
{}^1 \boldsymbol{V}_1 =& {}^1 X_{0} \ {}^0 \boldsymbol{V}_0 + {}^1 \boldsymbol{S}_1 \dot{q}_1 = {}^1 \boldsymbol{S}_1 \dot{q}_1 \\
{}^2 \boldsymbol{V}_2 =& {}^2 X_{1} \ {}^1 \boldsymbol{V}_1 + {}^2 \boldsymbol{S}_2 \dot{q}_2 \\
{}^3 \boldsymbol{V}_3 =& {}^3 X_{2} \ {}^2 \boldsymbol{V}_2 + {}^3 \boldsymbol{S}_3 \dot{q}_3 \\
\end{aligned}
$$

各リンクの６次元ベクトルと$6 \times 6$ 行列の演算を行えばよく、全てのリンクの速度式の展開を求める必要がない。これがForward Recursionの利点である。



## 3. Spatial Accelerationの再帰計算 (Forward Recursion)

2節で導出したSpatical Velocityの時間微分からSpatial Accelerationの再帰式を導出する。

子リンクのSpatial Velocityは以下の再帰式である。

$$
{^i}\boldsymbol{V}_{i} = {^i}X_{i-1} \ {^{i-1}}\boldsymbol{V}_{i-1} + {^i}\boldsymbol{S}_{i} \dot{q}_i
$$

ここで以下を定義する。

$$
{}^i \boldsymbol{V}_{i-1} = {^i}X_{i-1} \ {^{i-1}}\boldsymbol{V}_{i-1} \space , \quad {}^i \boldsymbol{V}_{J_i} = {^i}\boldsymbol{S}_{i} \dot{q}_i
$$

すると子リンクのSpatical Velocityは親リンクの運動と関節の相対運動を合成で表現できる。

$$
{^i}\boldsymbol{V}_{i} = {}^i \boldsymbol{V}_{i-1} + {}^i \boldsymbol{V}_{J_i}
$$

上式を子リンク座標$i$の時間微分を行い、[このNote](Spatial_Acceleration_and_Cross_Product.ipynb)の2節で、子リンク$i$の運動を親リンク$i-1$の相対運動として以下で表現できることを導いた。これはすべての子リンク座標系$i$で示され、同じ基準点で表現されている。この式が再帰式の基本となる。

$$
\boxed{
{}^i \boldsymbol{A}_i = {^i}\boldsymbol{A}_{i-1} + \frac{d}{dt}({^i}\boldsymbol{V}_{i/(i-1)}) + \text{crm}({^i}\boldsymbol{V}_i){^i}\boldsymbol{V}_{i/(i-1)}
}
$$

- ${^i}\boldsymbol{A}_{i-1}$ : 親リンク$i-1$のSpatial Accelration を子リンク座標系$i$で表したもの。
- $\frac{d}{dt}({^i}\boldsymbol{V}_{i/(i-1)})$ : 親$i-1$からの子$i$の相対 Spatial Velocity の子座標系$i$における時間微分
- $\text{crm}({^i}\boldsymbol{V}_{i})$ : 子リンク$i$の Spatial Velocity によるSpatial Velocity Cross Product
- ${^i}\boldsymbol{V}_{i/(i-1)}$ : 親$i-1$からの子$i$の相対 Spatial Velocity 

${^i}\boldsymbol{A}_{i-1}$ は[このNote](Spatial_Acceleration_and_Cross_Product.ipynb)の1.2.6で導出した Spatial AccelerationのSpatial Transformを用いてSpatial Velocityと同じ変換行列を用いて以下で表現できる。

$$
{^i}\boldsymbol{A}_{i-1} = {}^i X _{i-1} {^{i-1}}\boldsymbol{A}_{i-1}
$$

${^i}\boldsymbol{V}_{i/(i-1)}$は相対 Spatial Velocityであるので、Motion Subspaceを用いて以下で記述できる。

$$
{^i}\boldsymbol{V}_{i/(i-1)} = {^i}\boldsymbol{S}_{i} \dot{q}_i
$$

$\frac{d}{dt}({^i}\boldsymbol{V}_{i/(i-1)})$ は1自由度の回転関節または直動関節を対象とし、Motion Subspaceは子リンク座標において一定であるため、以下となる。

$$
\frac{d}{dt}({^i}\boldsymbol{V}_{i/(i-1)}) = \frac{d}{dt}({^i}\boldsymbol{S}_{i} \dot{q}_i) = {^i}\boldsymbol{S}_{i} \ddot{q}_i
$$

これを上式に代入すると、次のSpatial Accelerationの再帰式となる。

$$
\boxed{
{}^i \boldsymbol{A}_i = {}^i X _{i-1} {^{i-1}}\boldsymbol{A}_{i-1} + {^i}\boldsymbol{S}_{i} \ddot{q}_i + \text{crm}({^i}\boldsymbol{V}_i){^i}\boldsymbol{S}_{i} \dot{q}_i
}
$$

### 3.1. crm の必要性 

$\text{crm}({^i}\boldsymbol{V}_i){^i}\boldsymbol{S}_{i} \dot{q}_i$ について考える。


Spatial Velocity Cross Prodcustは ${^i}\boldsymbol{V}_i$の要素を用いて以下である。ここで${}^i \boldsymbol{V}_{i}$ は親リンクからの相対運動ではなく、親リンクの運動も含んだ子リンクの運動である。

$$
\text{crm}({^i}\boldsymbol{V}_i) = \begin{bmatrix} 
[{}^i \boldsymbol{\omega}_{i}]^\land & 0 \\
[{}^i\boldsymbol{v}_i]^\land & [{}^i \boldsymbol{\omega}_{i}]^\land
\end{bmatrix}
$$

子リンクが$z_i$軸まわりに$\dot{q}_i$の運動を行う場合以下となる。これは親リンクからの子リンクの相対運動である。

$$
\boldsymbol{S}_{i} \dot{q}_i = \begin{bmatrix} 0 & 0 & \dot{q}_i & 0 & 0 & 0 \end{bmatrix}^T
$$

したがって、次のように親リンクの運動を含んだ子リンクの角速度(${}^i \boldsymbol{\omega}_{i}$)と並進運動(${}^i\boldsymbol{v}_i$)により、子リンク自身の相対運動が変化するという、速度依存の加速度を表している。

$$
\text{crm}({^i}\boldsymbol{V}_i){^i}\boldsymbol{S}_{i} \dot{q}_i = \begin{bmatrix}
{}^i \boldsymbol{\omega}_{i} \times (e_z \dot{q}_i) \\ {}^i\boldsymbol{v}_i \times (e_z \dot{q}_i)
\end{bmatrix}
$$

関節加速度$\ddot{q}_i$がゼロであっても、親リンクと子リンクが運動していればこの項はゼロとは限らない。

これが単純に親リンクと加速度と子リンクの相対加速度のみの加算で子リンクの加速度を表せない理由である。

$$
{}^i \boldsymbol{A}_i = {}^i X _{i-1} {^{i-1}}\boldsymbol{A}_{i-1} + {^i}\boldsymbol{S}_{i} \ddot{q}_i
$$

### 3.2. 再帰計算について

固定ベースをリンク0とすると、Spatial Velocityは以下となる。

$$
{}^0 \boldsymbol{V}_0 = 0
$$

重力をいったん無視すれば、リンク0のSpatial Accelrationは以下となる。

$$
{}^0 \boldsymbol{A}_0 = 0
$$

例えば2リンクアームの場合、Spatial Velcityの再帰式とSpatial Accelrationの再帰式より、次のように順番に計算を行うことが出来る。

- リンク1

$$
\begin{aligned}
{}^1 \boldsymbol{V}_1 &= {^1}X_{0} \ {}^0 \boldsymbol{V}_0 + {^1}\boldsymbol{S}_1 \dot{q}_1 \\
& = {^1}\boldsymbol{S}_1 \dot{q}_1 \\
{}^1 \boldsymbol{A}_1 &= {}^1 X _{0} {^{0}}\boldsymbol{A}_{0} + {^1}\boldsymbol{S}_{1} \ddot{q}_1 + \text{crm}({^1}\boldsymbol{V}_1){^1}\boldsymbol{S}_{1} \dot{q}_1 \\
&= {^1}\boldsymbol{S}_{1} \ddot{q}_1 + \text{crm}({^1}\boldsymbol{S}_1 \dot{q}_1){^1}\boldsymbol{S}_{1} \dot{q}_1 \\
& ={^1}\boldsymbol{S}_{1} \ddot{q}_1
\end{aligned}
$$ 

- リンク2

$$
\begin{aligned}
{}^2 \boldsymbol{V}_2 &= {^2}X_{1} \ {}^1 \boldsymbol{V}_1 + {^2}\boldsymbol{S}_2 \dot{q}_2 \\
{}^2 \boldsymbol{A}_2 &= {}^2 X _{1} {^{1}}\boldsymbol{A}_{1} + {^2}\boldsymbol{S}_{2} \ddot{q}_2 + \text{crm}({^2}\boldsymbol{V}_2){^2}\boldsymbol{S}_{2} \dot{q}_2 \\
\end{aligned}
$$ 


## 4. Spatial Force

[このNote](Spatial_Newton_Euler.ipynb)では以下のSpatial Newton-Eueler方程式を導出した。この式は剛体に固定された任意の点$O$を基準点とし、剛体座標$\Sigma_B$で表した式である。

$$
{^B}\boldsymbol{f}_O = {^B}\mathcal{I}_O {^B}{\boldsymbol{A}}_O + \text{crf}({^B}\boldsymbol{V}_O){^B}\mathcal{I}_O {^B}\boldsymbol{V}_O
$$

この式をRNEAへ適用する。基準点は各リンクの座標系$\Sigma_i$の原点$O_i$とし、リンク座標系で表すと以下である。

$$
{^i}\boldsymbol{f}_i^{\text{net}} = {^i}\mathcal{I}_i {^i}{\boldsymbol{A}}_i + \text{crf}({^i}\boldsymbol{V}_i){^i}\mathcal{I}_i {^i}\boldsymbol{V}_i
$$

- ${^i}\boldsymbol{f}_i^{\text{net}}$ : リンク$i$単体の運動を満たすために発生すべき正味のSpatial Force Vector(慣性項) をリンク座標系$\Sigma_i$で示したもの(基準点$O_i$)
- ${^i}\mathcal{I}_i$ : リンク$i$のSpatial Inertiaをリンク座標系$\Sigma_i$で示したもの(基準点$O_i$)
- ${^i}{\boldsymbol{A}}_i$ : リンク$i$のSpatial Accelerationをリンク座標系$\Sigma_i$で示したもの(基準点$O_i$)
- ${^i}\boldsymbol{V}_i$ : リンク$i$のSpatial Velocityをリンク座標系$\Sigma_i$で示したもの(基準点$O_i$)


リンク$i$の運動である、${^i}\boldsymbol{V}_i, {^i}{\boldsymbol{A}}_i$は再帰計算によりリンクのベースから先端に向かって順に計算される。これらのSpatial Motion はリンクの相対運動ではなく、親リンクの運動も加味されたものである。

しかし、リンク座標系 $\Sigma_i$ で示された Spatial Inertia ${^i}\mathcal{I}_i$ はリンク$i$単体のSpatial Inertiaである。そのため、上式で計算する　${^i}\boldsymbol{f}_i^{\text{net}}$ はリンク$i$単体の運動を成立させるために必要な正味のSpatial Force Vector(慣性項)である。

一方、関節トルクはその関節が駆動するリンクだけでなく、子・孫リンクの運動に必要なSpatial Forceも考慮して求める必要がある。<br>
例えば3リンク構造のアームの関節1ではリンク1だけでなく、リンク2とリンク3に必要なSpatial Force Vectorも関節1に伝達される。
したがって、RENAでは各リンク単体のSpatial Force Vectorを計算した後、Backward Recursionによって子リンクから親リンクへSpatial Force Vectorを伝播・加算し、Motion Subspace を用い関節トルクを求める。



## 5. Backward Recursion

4節で計算したリンク$i$単体の正味のSpatial Force(慣性項)は以下である。

$$
{^i}\boldsymbol{f}_i^{\text{net}} = {^i}\mathcal{I}_i {^i}{\boldsymbol{A}}_i + \text{crf}({^i}\boldsymbol{V}_i){^i}\mathcal{I}_i {^i}\boldsymbol{V}_i
$$

ここで次の記号を定義する。

- ${}^i \boldsymbol{f}_{(i-1) \rightarrow i}$ : 親リンク$i-1$からリンク$i$に作用するSpatial Force
- ${}^i \boldsymbol{f}_{(i+1) \rightarrow i}$ : 子リンク$i+1$からリンク$i$に作用するSpatial Force

リンク$i$単体の運動を成立させるために必要な慣性項${^i}\boldsymbol{f}_i^{\text{net}}$であり、親リンク$i-1$と子リンク$i+1$から受けるSpatial Force は外力となる。よって以下の釣り合いの式が成り立つ。これはツリー構造で子リンクが複数ない場合である。

$$
{^i}\boldsymbol{f}_i^{\text{net}} = {}^i \boldsymbol{f}_{(i-1) \rightarrow i} + {^i} \boldsymbol{f}_{(i+1) \rightarrow i}
$$

これは質点の運動を考えると、目標加速度となる運動と外力の合計が成り立つ以下の式と同等の構造である。

$$
m \boldsymbol{a} = \sum \boldsymbol{F}_{\text{ext}}
$$

### 5.1. 関節伝達力

子リンクから親リンクへの関節伝達力を考える。ここでは二つのリンクを持つシステムを考える。<br>
親リンクをP、子リンクをCとして、親リンク$P$から子リンク$C$に作用するSpatial Forceを子リンク座標系で表すと以下である。

$$
{}^{C} \boldsymbol{f}_{P \rightarrow C}
$$

子リンク$C$から親リンク$P$に作用するSpatial Forceを子リンク座標系で表すと以下の式の左辺である。これは作用・反作用より、親リンク$P$から子リンク$C$へ作用するSpatial Forceの符号が反転したものである。

$$
{}^{C} \boldsymbol{f}_{C \rightarrow P} = - {}^{C} \boldsymbol{f}_{P \rightarrow C}
$$

[このNote](./spatial_vector.ipynb)の3.4節では、親リンク$P$から子リンク$C$へのSpatial Force TransformをSpatial Velocityの変換行列と関連させ以下のように導いた。以下は座標と基準点の変換だが、右下の$P$はリンク$P$の運動と力であることを表している。

$$
{}^{C}\boldsymbol{V}_{P} = {}^{C} X _{P} {^P}\boldsymbol{V}_{P} \quad \rightarrow \quad {}^{C}\boldsymbol{f}_{P} = \left({}^{C} X _{P}\right)^{-T} {^P}\boldsymbol{f}_{P}
$$

これより、子リンク$C$から親リンク$P$への座標系・基準点の変換は、変換行列の逆行列を用いて以下となる。これは、子リンク$C$から親リンク$P$に作用するSpatial Force を親リンク$P$で表したものである。

$$
{}^{P} \boldsymbol{f}_{C \rightarrow P} = - \left({}^{C} X _{P}\right)^{T} {}^{C} \boldsymbol{f}_{P \rightarrow C}
$$

上式の関係を$C$を子リンク$i+1$、$P$をリンク$i$とすると以下の式になる。

$$
{}^{i} \boldsymbol{f}_{(i+1) \rightarrow i} = - \left({}^{i+1} X _{i}\right)^{T} {}^{i+1} \boldsymbol{f}_{i \rightarrow (i+1)}
$$

これを力の釣り合いに代入すると、以下となる。

$$
{^i}\boldsymbol{f}_i^{\text{net}} = {}^i \boldsymbol{f}_{(i-1) \rightarrow i} - \left({}^{i+1} X _{i}\right)^{T} {}^{i+1} \boldsymbol{f}_{i \rightarrow (i+1)}
$$

これより、親リンク$i-1$からリンク$i$へ作用する関節伝達力は、リンク$i$単体の慣性項と、子リンク$i+1$へ伝達するSpatial親リンク座標$\Sigma_i$で表現する式は以下となる。

$$
{}^i \boldsymbol{f}_{(i-1) \rightarrow i} = {^i}\boldsymbol{f}_i^{\text{net}} + \left({}^{i+1} X _{i}\right)^{T} {}^{i+1} \boldsymbol{f}_{i \rightarrow (i+1)}
$$

### 5.2 Backward Recursion の定式化

親リンク$i-1$からリンク$i$に作用するSpatial Force を以下のように定義する。

$$
{^i} \boldsymbol{f}_i \equiv {}^i \boldsymbol{f}_{(i-1) \rightarrow i}
$$

同様に、リンク$i$から子リンク$i+1$に作用するSpatial Force を以下で定義する。

$$
{}^{i+1} \boldsymbol{f}_{i+1} \equiv {}^{i+1} \boldsymbol{f}_{i \rightarrow (i+1)}
$$

これらを5.1 で導出した式に代入すると、以下の再帰式となる。

$$
{}^i \boldsymbol{f}_{i} = {^i}\boldsymbol{f}_i^{\text{net}} + \left({}^{i+1} X _{i}\right)^{T} {}^{i+1} \boldsymbol{f}_{i+1}
$$

これがRNEAにおけるBackward Recusionの基本式である。<br>
この式は、リンク$i$単体の慣性項に、子リンク$i+1$の関節伝達力をリンク座標系へ変換して加算することで、リンク$i$の関節伝達力を計算することを意味する。

### 5.3. 3リンクアームの場合の例

リンク1,2,3からなる直列アームを考える。リンク1は固定ベース0と連結されているとする。各リンクの関節にあるモーター(アクチュエータ)のステーター(固定側)はそのリンクの親側に、ローター(回転側)はそのリンクにあると考える。

リンク3は、関節3を介して親リンク2から力とモーメントを受ける。この関節伝達力には、アクチュエーターが発生する駆動トルクと、関節の拘束によって生じる力・モーメントが含まれる。先端リンク3には子リンクが存在しないため、リンク3が親リンク2から受ける関節伝達力は、リンク3単体の慣性項に等しくなる。

$$
{}^3 \boldsymbol{f}_3 = {}^3 \boldsymbol{f}_3^{\text{net}}
$$

リンク3を動かすためにリンク2は${}^3 \boldsymbol{f}_3$の関節伝達力をリンク3へ作用させた。作用・反作用によりリンク2はリンク
3から以下の作用力を受ける。

$$
{^2}\boldsymbol{f}_{3 \rightarrow 2} = - \left({}^{3} X _{2}\right)^{T} {}^{3} \boldsymbol{f}_{3}
$$

一方、リンク2がリンク1から受ける関節伝達力(左辺)は、リンク2単体の慣性項(右辺第一項)と、リンク3がリンク2から受ける関節伝達力をリンク2の座標系・基準点へ変換したもの(右辺第二項)の和となる。リンク2がリンク3から受ける反力は上記であるが、右辺第二項の符号はその逆となっている。つまり右辺第二項は反力を補償するために必要な力・モーメントである。

$$
{}^2 \boldsymbol{f}_{2} = {^2}\boldsymbol{f}_2^{\text{net}} + \left({}^{3} X _{2}\right)^{T} {}^{3} \boldsymbol{f}_{3}
$$

最後のリンク1では以下となる。リンク1がリンク0から受ける関節伝達力(左辺)はリンク1単体の慣性項(右辺第一項)とリンク2がリンク1から受ける関節伝達力をリンク1の座標系・基準点へ変換した補償のための力・モーメント(右辺第二項)の和となる。右辺第二項の${}^{2} \boldsymbol{f}_{2}$にはリンク3の運動に必要な関節伝達力も含まれているため、リンク1がリンク0からうける関節伝達力には、リンク1,2,3の運動の影響が反映される。

$$
{}^1 \boldsymbol{f}_{1} = {^1}\boldsymbol{f}_1^{\text{net}} + \left({}^{2} X _{1}\right)^{T} {}^{2} \boldsymbol{f}_{2}
$$

### 5.4. Motion Subspaceによる関節トルクの計算

5.3 で関節伝達力の計算を手先リンクからベースの方向へ順番に計算を行い、それぞれリンクで求めた。これに1節で求めた、Spatial ForceとMotion Subspaceの関係からそれぞれのアクチュエータで発生すべきトルクを計算する。

$$
\tau_i = \left({}^i \boldsymbol{S}_i \right)^T {}^i\boldsymbol{f}_i
$$

例えばリンク1のMotion Subspaceを以下とする。

$$
{^1}\boldsymbol{S}_{1} = \begin{bmatrix} 0 \\ 0 \\ 1 \\ 0 \\ 0 \\ 0 \end{bmatrix}
$$

Backward Recursion の計算でリンク1がリンク0から受けるSpatial Force は以下であったとする。

$$
{^1}\boldsymbol{f}_{1} = \begin{bmatrix} 1 \\ -5 \\ 10 \\ 3 \\ -1 \\ 20 \end{bmatrix}
$$

リンク1用のアクチュエータで発生すべきトルク$\tau_1$は以下で計算できる。

$$
\tau_1 = \left({}^1 \boldsymbol{S}_1 \right)^T {}^1\boldsymbol{f}_1 = 10 \space \text( [Nm])
$$




## 6. 重力と外力の考慮

### 6.1 重力の考慮

リンク$i$の重心に作用する重力を以下とする。

$$
{}^i \boldsymbol{F}_{g, i} = m_i \ {}^i\boldsymbol{g}
$$

この重力をリンク$i$の原点$O_i$で表して Spatial Forceは以下である。

$$
{^i}\boldsymbol{f}_{g,i} = \begin{bmatrix} {}^i \boldsymbol{n}_{g,i} \\ {}^i \boldsymbol{F}_{g, i}\end{bmatrix} = \begin{bmatrix} {}^i \boldsymbol{r}_{O_i G_i} \times m_i \ {}^i\boldsymbol{g} \\ m_i \ {}^i\boldsymbol{g} \end{bmatrix}
$$

Backward Recustionでは、このSpatial Forceを補償するための関節伝達力を計算するため、次の再帰式となる。

$$
{}^i \boldsymbol{f}_{i} = {^i}\boldsymbol{f}_i^{\text{net}} + \left({}^{i+1} X _{i}\right)^{T} {}^{i+1} \boldsymbol{f}_{i+1} - {^i}\boldsymbol{f}_{g,i}
$$

#### 6.1.1 単体リンクの慣性項への重力の組み込み

世界座標系の重力加速度を以下とする。

$$
{}^0\boldsymbol{g} = \begin{bmatrix} g_x \\ g_y \\ g_z \end{bmatrix}
$$

Spatial Acceleartion として重力加速度を表すと、

$$
{}^0 \boldsymbol{G} = \begin{bmatrix} 0 \\ {}^0 \boldsymbol{g} \end{bmatrix}
$$

リンク$i$の座標系・原点へ変換すると、Spatial Transformを用いて以下となる。

$$
{}^i \boldsymbol{G}_i = {}^i X_ {0} {}^0 \boldsymbol{G} = \begin{bmatrix} 0 \\ {}^i \boldsymbol{g} \end{bmatrix}
$$

ここでリンク$i$の基準点$O_i$におけるSpatial Inertiaをリンク$i$座標は以下である。

$$
{}^i \mathcal{I}_i = 
\begin{bmatrix}
{}^i \boldsymbol{I}_{G_i} - m_i [{}^i \boldsymbol{r}_{O_i G_i }]^\land [{}^i \boldsymbol{r}_{O_i G_i}]^\land & m_i [{}^i \boldsymbol{r}_{O_i G_i }]^\land \\
- m_i [{}^i \boldsymbol{r}_{O_i G_i}]^\land & m_i \boldsymbol{I}_3 
\end{bmatrix} \in \mathbf{R}^{6 \times 6}
$$

このSpatial Inerti と重力のSpatial Accelerationを掛けると、Spatial Forceとなる。

$$
{}^i \mathcal{I}_i {}^i \boldsymbol{G}_i = \begin{bmatrix} {}^i \boldsymbol{r}_{O_i G_i } \times m_i \ {}^i \boldsymbol{g} \\ m_i \ {}^i \boldsymbol{g}\end{bmatrix} = {^i}\boldsymbol{f}_{g,i}
$$

リンク単体のNewton-Euler法の方程式にSpatial Force を補償として組み込むと以下となる。

$$
\begin{aligned}
{^i}\boldsymbol{f}_i^{\text{net}} &= {^i}\mathcal{I}_i {^i}{\boldsymbol{A}}_i + \text{crf}({^i}\boldsymbol{V}_i){^i}\mathcal{I}_i {^i}\boldsymbol{V}_i - {}^i \mathcal{I}_i {}^i \boldsymbol{G}_i \\
&= {^i}\mathcal{I}_i \left( {^i}{\boldsymbol{A}}_i - {}^i \boldsymbol{G}_i \right) + \text{crf}({^i}\boldsymbol{V}_i){^i}\mathcal{I}_i {^i}\boldsymbol{V}_i
\end{aligned}
$$

これより重力をSpatial Forceとして差し引くことは、Spatial Accelerationから重力加速度を差し引くこととなる。

#### 6.1.2 Forward Recustionでの重力の考慮

通常の固定ベースのSpatial Accelerationでは、次のようにゼロを設定する。

$$
{}^0 \boldsymbol{A}_0 = 0
$$

これを、重力加速度を差し引いたSpatial Accelerationとして次のように初期化する。これは実際のベース加速度ではなく、重力補償を組み込んだ計算用のSpatial Accelerationである。

$$
{}^0 \widetilde{\boldsymbol{A}}_0 = \begin{bmatrix} 0 \\ -{}^0 \boldsymbol{g} \end{bmatrix}
$$

Forward RecustionのSpatil Accelerationの計算では $\widetilde{\boldsymbol{A}}$ を用いた計算を行う。

$$
\boxed{
{}^i \widetilde{\boldsymbol{A}}_i = {}^i X _{i-1} {^{i-1}}\widetilde{\boldsymbol{A}}_{i-1} + {^i}\boldsymbol{S}_{i} \ddot{q}_i + \text{crm}({^i}\boldsymbol{V}_i){^i}\boldsymbol{S}_{i} \dot{q}_i
}
$$

各リンクのSpatial Acceleartionは Spatial Transformを通じて$-\boldsymbol{g}$が再帰的に伝播する。

$$
{}^i \widetilde{\boldsymbol{A}}_i = {}^i {\boldsymbol{A}}_i - {}^i \boldsymbol{G}_i
$$


慣性項の計算では $\widetilde{\boldsymbol{A}}$ を用いた計算により、重力の補償項とした現れる。

$$
\begin{aligned}
{^i}\widetilde{\boldsymbol{f}}_i^{\text{net}} &= {^i}\mathcal{I}_i \ {^i}\widetilde{\boldsymbol{A}}_i + \text{crf}({^i}\boldsymbol{V}_i){^i}\mathcal{I}_i {^i}\boldsymbol{V}_i \\
&= {^i}\mathcal{I}_i \ \left( {^i}{\boldsymbol{A}}_i + {}^i \boldsymbol{G}_i \right)+ \text{crf}({^i}\boldsymbol{V}_i){^i}\mathcal{I}_i {^i}\boldsymbol{V}_i \\
&= {^i}{\boldsymbol{f}}_i^{\text{net}} + {}^i \boldsymbol{f}_{g,i}
\end{aligned}
$$

#### 6.1.3 固定ベースへの重力加速度の設定例

例えば重力が世界座標の-$z$方向なら、

$$
{}^0 \boldsymbol{g} = \begin{bmatrix} 0 & 0 & -9.81 \end{bmatrix}
$$

よって、ベースを以下のSpatial Accelerationで初期化すればよい。

$$
{}^0 \widetilde{\boldsymbol{A}}_0 = \begin{bmatrix} 0 & 0 & 0 & 0 & 0 & 9.81 \end{bmatrix}
$$

### 6.2 外力の考慮

各リンク$i$に加わる外力${}^i \boldsymbol{f}_{\text{ext}, i}$ が加わったとして、それを補償して運動を行うためにBackward Recursionで次のように計算する。

$$
{}^i \boldsymbol{f}_{i} = {^i}\boldsymbol{f}_i^{\text{net}} + \left({}^{i+1} X _{i}\right)^{T} {}^{i+1} \boldsymbol{f}_{i+1} - {}^i \boldsymbol{f}_{\text{ext}, i}
$$

例えば以下の図のように、先端リンク$e$が$x_0$方向に物体を押しているとき、アーム先端が受ける反力は以下である。

$$
{}^0 \boldsymbol{F}_{\text{ext}} = \begin{bmatrix} -\mu m g \\ 0 \\ 0 \end{bmatrix}
$$

この力がセンサにより先端リンク座標系$e$の点$P$で得られているとする。すると、先端リンク$e$の座標系・基準点におけるSpatial Forceは次のようになる。

$$
{}^e \boldsymbol{f}_{\text{ext}, e} = \begin{bmatrix} {}^e \boldsymbol{r}_{O_e P} \times {}^e \boldsymbol{F}_{\text{ext}} \\ {}^e \boldsymbol{F}_{\text{ext}}
\end{bmatrix}
$$

これをBackward Recursion の先端リンク$e$の計算で次のように計算する。後の子リンクからベース方向への再帰計算は通常通り行う。

$$
{}^e \boldsymbol{f}_e = {}^e \boldsymbol{f}_e^{^text{net}} - {}^e \boldsymbol{f}_{\text{ext}, e}
$$ 

<p align="center">
<img src="./images/external_force_example_image.png" style="width:50%;"/>
</p>



## 7. RNEA の計算の流れ

これまで導出した式を用いて、RNEA（Recursive Newton–Euler Algorithm）の計算手順を整理する。

RNEAは、関節位置 $\boldsymbol q$、関節速度 $\dot{\boldsymbol q}$、関節加速度 $\ddot{\boldsymbol q}$ を入力として、指定された運動を実現するために必要な関節駆動力 $\boldsymbol\tau$ を計算する逆動力学アルゴリズムである。

### 7.1. 初期化

固定ベースをリンク0とし、ベースのSpatial Velocityをゼロとする。

$$
{}^0\boldsymbol V_0=\boldsymbol 0
$$

重力加速度を ${}^0\boldsymbol g$ とすると、重力を考慮するためにベースのSpatial Accelerationを以下のように初期化する。

$$
{}^0\widetilde{\boldsymbol A}_0=
\begin{bmatrix}
\boldsymbol 0\\
-{}^0\boldsymbol g
\end{bmatrix}
$$

ここでの ${}^0 \widetilde{\boldsymbol A}_0$ は、重力を組み込んだ計算用の値であり、実際のベースの加速度ではない。

### 7.2. Forward Recursion

ベース側のリンク1から先端リンク $N$ へ向かって、各リンクのSpatial VelocityとSpatial Accelerationを計算する。

**Spatial Velocity**

$$
{}^i\boldsymbol V_i
=
{}^iX_{i-1}{}^{i-1}\boldsymbol V_{i-1}
+
{}^i\boldsymbol S_i\dot q_i
$$

**Spatial Acceleration**

$$
{}^i\widetilde{\boldsymbol A}_i
=
{}^iX_{i-1}{}^{i-1}\widetilde{\boldsymbol A}_{i-1}
+
{}^i\boldsymbol S_i\ddot q_i
+
\operatorname{crm}({}^i\boldsymbol V_i)
{}^i\boldsymbol S_i\dot q_i
$$

続いて、各リンクのSpatial Inertiaを用いて、重力を組み込んだ慣性項を計算する。

$$
{}^i \widetilde{\boldsymbol f}_i^{\mathrm{net}}
=
{}^i\mathcal I_i{}^i \widetilde{\boldsymbol A}_i
+
\operatorname{crf}({}^i\boldsymbol V_i)
{}^i\mathcal I_i{}^i\boldsymbol V_i
$$

なお、この節の ${}^i \widetilde{\boldsymbol A}_i $ と ${}^i \widetilde{\boldsymbol f}_i^{\mathrm{net}}$ は、重力を初期条件に組み込んだ計算用の量である。

### 7.3. Backward Recursion

先端リンク $N$ からベース側のリンク1へ向かって、関節伝達力を計算する。

先端リンクには子リンクが存在しないため、

$$
{}^N\boldsymbol f_N
=
{}^N\widetilde{\boldsymbol f}_N^{\mathrm{net}}
-
{}^N\boldsymbol f_{\mathrm{ext},N}
$$

となる。

それ以外のリンクでは、子リンクの関節伝達力を親リンクの座標系・基準点へ変換し、以下のように計算する。

$$
{}^i\boldsymbol f_i
=
{}^i \widetilde{\boldsymbol f}_i^{\mathrm{net}}
+
({}^{i+1}X_i)^T{}^{i+1}\boldsymbol f_{i+1}
-
{}^i\boldsymbol f_{\mathrm{ext},i}
$$

ここで、${}^i\boldsymbol f_{\mathrm{ext},i}$ は環境からリンク $i$ に作用する外力・外モーメントであり、リンク $i$ の座標系・基準点で表されている。

**関節駆動力**

各リンクの関節伝達力から、Motion Subspaceを用いて関節駆動力を計算する。

$$
\boxed{
\tau_i=({}^i\boldsymbol S_i)^T{}^i\boldsymbol f_i
}
$$

### 7.4. 計算手順のまとめ

1. 関節位置・速度・加速度 $\boldsymbol q,\dot{\boldsymbol q},\ddot{\boldsymbol q}$ を与える。
2. ベースのSpatial VelocityとSpatial Accelerationを初期化する。
3. **Forward Recursion**：各リンクのSpatial Velocity、Spatial Acceleration、慣性項をベースから先端へ計算する。
4. **Backward Recursion**：外力と子リンクの関節伝達力を考慮し、先端からベースへ関節伝達力を計算する。
5. Motion Subspaceにより各関節の駆動力を求める。

以上により、指定した運動を実現するために必要な関節駆動力が求まる。

以上